<a href="https://colab.research.google.com/github/SofiiaBobr/goit_machine_learning/blob/main/goit-algo-hw-05/task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import timeit
from pathlib import Path


def read_file(filename):
    """Reads a text file. Supports UTF-8 and CP1251 encodings."""
    path = Path(filename)

    for encoding in ("utf-8", "utf-8-sig", "cp1251"):
        try:
            return path.read_text(encoding=encoding)
        except UnicodeDecodeError:
            continue

    raise UnicodeDecodeError(
        "unknown",
        b"",
        0,
        1,
        f"Cannot decode file {filename}"
    )



def build_shift_table(pattern):
    table = {}
    pattern_length = len(pattern)

    for index in range(pattern_length - 1):
        table[pattern[index]] = pattern_length - index - 1

    return table


def boyer_moore_search(text, pattern):
    if pattern == "":
        return 0

    if len(pattern) > len(text):
        return -1

    shift_table = build_shift_table(pattern)
    index = 0

    while index <= len(text) - len(pattern):
        pattern_index = len(pattern) - 1

        while (
            pattern_index >= 0
            and text[index + pattern_index] == pattern[pattern_index]
        ):
            pattern_index -= 1

        if pattern_index < 0:
            return index

        last_char = text[index + len(pattern) - 1]
        index += shift_table.get(last_char, len(pattern))

    return -1


# =========================
# Knuth-Morris-Pratt algorithm
# =========================

def compute_lps(pattern):
    lps = [0] * len(pattern)
    length = 0
    index = 1

    while index < len(pattern):
        if pattern[index] == pattern[length]:
            length += 1
            lps[index] = length
            index += 1
        else:
            if length != 0:
                length = lps[length - 1]
            else:
                lps[index] = 0
                index += 1

    return lps


def kmp_search(text, pattern):
    if pattern == "":
        return 0

    if len(pattern) > len(text):
        return -1

    lps = compute_lps(pattern)

    text_index = 0
    pattern_index = 0

    while text_index < len(text):
        if text[text_index] == pattern[pattern_index]:
            text_index += 1
            pattern_index += 1

        if pattern_index == len(pattern):
            return text_index - pattern_index

        if (
            text_index < len(text)
            and text[text_index] != pattern[pattern_index]
        ):
            if pattern_index != 0:
                pattern_index = lps[pattern_index - 1]
            else:
                text_index += 1

    return -1



# Rabin-Karp algorithm


def rabin_karp_search(text, pattern):
    if pattern == "":
        return 0

    if len(pattern) > len(text):
        return -1

    base = 256
    modulus = 101

    pattern_length = len(pattern)
    text_length = len(text)

    pattern_hash = 0
    text_hash = 0
    highest_power = 1

    for _ in range(pattern_length - 1):
        highest_power = (highest_power * base) % modulus

    for index in range(pattern_length):
        pattern_hash = (
            base * pattern_hash + ord(pattern[index])
        ) % modulus
        text_hash = (
            base * text_hash + ord(text[index])
        ) % modulus

    for index in range(text_length - pattern_length + 1):
        if pattern_hash == text_hash:
            if text[index:index + pattern_length] == pattern:
                return index

        if index < text_length - pattern_length:
            text_hash = (
                base * (
                    text_hash - ord(text[index]) * highest_power
                )
                + ord(text[index + pattern_length])
            ) % modulus

            if text_hash < 0:
                text_hash += modulus

    return -1



# Time measuring


def measure_time(search_function, text, pattern, repeat_count=100):
    return timeit.timeit(
        lambda: search_function(text, pattern),
        number=repeat_count
    )


def find_fastest(results):
    return min(results, key=results.get)


def main():
    text_1 = read_file("article1.txt")
    text_2 = read_file("article2.txt")

    algorithms = {
        "Boyer-Moore": boyer_moore_search,
        "Knuth-Morris-Pratt": kmp_search,
        "Rabin-Karp": rabin_karp_search,
    }

    test_data = {
        "Article 1": {
            "text": text_1,
            "existing_pattern": "двійковий пошук",
            "fake_pattern": "квантовий суперпошук",
        },
        "Article 2": {
            "text": text_2,
            "existing_pattern": "розгорнутий список",
            "fake_pattern": "квантовий суперпошук",
        },
    }

    repeat_count = 100
    all_results = {}

    print("Substring search algorithms comparison")
    print("=" * 50)
    print(f"Repeat count for each measurement: {repeat_count}")

    for article_name, data in test_data.items():
        print(f"\n{article_name}")
        print("-" * 50)

        all_results[article_name] = {}

        for pattern_type, pattern in {
            "Existing pattern": data["existing_pattern"],
            "Fake pattern": data["fake_pattern"],
        }.items():
            print(f"\n{pattern_type}: {pattern}")

            pattern_results = {}

            for algorithm_name, algorithm_function in algorithms.items():
                execution_time = measure_time(
                    algorithm_function,
                    data["text"],
                    pattern,
                    repeat_count
                )

                position = algorithm_function(data["text"], pattern)
                pattern_results[algorithm_name] = execution_time

                print(
                    f"{algorithm_name:20} | "
                    f"time: {execution_time:.6f} sec | "
                    f"position: {position}"
                )

            fastest = find_fastest(pattern_results)
            all_results[article_name][pattern_type] = pattern_results

            print(f"Fastest algorithm: {fastest}")

    print("\nOverall result")
    print("-" * 50)

    total_results = {
        "Boyer-Moore": 0,
        "Knuth-Morris-Pratt": 0,
        "Rabin-Karp": 0,
    }

    for article_results in all_results.values():
        for pattern_results in article_results.values():
            for algorithm_name, execution_time in pattern_results.items():
                total_results[algorithm_name] += execution_time

    for algorithm_name, total_time in total_results.items():
        print(f"{algorithm_name:20} | total time: {total_time:.6f} sec")

    print(f"\nOverall fastest algorithm: {find_fastest(total_results)}")


if __name__ == "__main__":
    main()

Substring search algorithms comparison
Repeat count for each measurement: 100

Article 1
--------------------------------------------------

Existing pattern: двійковий пошук
Boyer-Moore          | time: 0.002766 sec | position: 304
Knuth-Morris-Pratt   | time: 0.011536 sec | position: 304
Rabin-Karp           | time: 0.012076 sec | position: 304
Fastest algorithm: Boyer-Moore

Fake pattern: квантовий суперпошук
Boyer-Moore          | time: 0.042689 sec | position: -1
Knuth-Morris-Pratt   | time: 0.536431 sec | position: -1
Rabin-Karp           | time: 1.074152 sec | position: -1
Fastest algorithm: Boyer-Moore

Article 2
--------------------------------------------------

Existing pattern: розгорнутий список
Boyer-Moore          | time: 0.005359 sec | position: 1078
Knuth-Morris-Pratt   | time: 0.039029 sec | position: 1078
Rabin-Karp           | time: 0.041715 sec | position: 1078
Fastest algorithm: Boyer-Moore

Fake pattern: квантовий суперпошук
Boyer-Moore          | time: 0.064533 